# Phase 0.6 follow-ups on Colab — F2 (camera embedding) and F3 (idle-trimmed data)

Companion to `Details/phase06_camera_ablation_training.md` §8. Everything is pulled from the Hub;
nothing local is needed.

| | What it tests | Runs |
|---|---|---|
| **F2** | Is `top+wrist`'s reach deficit an *arbitration* problem? ACT gives both cameras identical positional embeddings, so the model can only tell the streams apart by appearance. Add a learned per-camera identity vector and retrain. | 1 × 60k (`both`) |
| **F3** | Does idle-trimming the training data let short action chunks work? The untrimmed set forced `n_action_steps=100`, i.e. ~9 observations per 30 s episode, which blunts any camera ablation. | 3 × 60k (`top`, `wrist`, `both`) |

**The recipe is identical to the 60k baselines** — same 45/5 balanced holdout, same seed 1000, same
`batch_size=8`, `eval_steps=5000`, fp32, no augmentation — so the new runs are directly comparable to
`act_top_s1000` / `act_wrist_s1000` / `act_both_s1000` in the same wandb project.

## Which Colab GPU?

**Pick L4.** The workload is small-convolution compute-bound, not memory-bound: ACT at batch 8 with two
cameras peaks at **3.5 GiB**, so an A100's 40 GB is pure waste, and a bigger batch buys nothing (measured
locally: throughput is flat at 58–60 samples/s from batch 8 to 32).

Estimates scaled from a measured RTX 5080 Laptop baseline (fp32: 5.3 it/s two-camera, 10.5 single):

| GPU | VRAM | est. it/s (2 cam) | F2 total | F3 total | F2+F3 |
|---|---|---|---|---|---|
| T4 | 16 GB | ~1.6 | ~10.5 h | ~21 h | **~32 h** — impractical, risks session limits |
| **L4** | **24 GB** | **~3.7** | **~4.5 h** | **~9 h** | **~13.5 h** ← recommended |
| A100 | 40 GB | ~6.4 | ~2.6 h | ~5.2 h | ~7.8 h, at ~2.5× the compute units per hour |

Roughly, per compute unit: L4 ≈ 65 units for both experiments, A100 ≈ 92, T4 ≈ 57 but spread over days.
**L4 is the best time-per-unit trade** and has ample VRAM. Check current Colab rates — they change.

> These are estimates. **The benchmark cell below measures your actual throughput** and prints real
> projections before you commit to a long run.

**Session limits matter.** Each individual run fits in a typical Colab session, but F3's three runs do
not. Every run below pushes checkpoints to the Hub, so a disconnect is recoverable — see §9.

## 1 — Runtime check

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"
print("torch", torch.__version__, "| cuda", torch.version.cuda, "|", name)
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime > Change runtime type > GPU (L4).")
if "T4" in name:
    print("\n!! T4 detected. F2+F3 would take ~32 h here. Runtime > Change runtime type > L4.")
elif "A100" in name:
    print("\nNote: A100 works but costs ~2.5x the compute units for ~1.7x the speed; L4 is the better trade.")

## 2 — Install

Installed from the **v0.6.1 source tree** rather than the wheel, because F2 patches `modeling_act.py`.
Takes a few minutes.

In [ ]:
!git clone --depth 1 --branch v0.6.1 https://github.com/huggingface/lerobot.git /content/lerobot
%pip install -q -e /content/lerobot
%pip install -q wandb
import lerobot; print("lerobot", lerobot.__version__)

> If `import lerobot` fails right after the install, restart the runtime (Runtime > Restart session) and re-run this cell — Colab caches the old module table.

## 3 — Authentication

No proxy here, unlike the local machine.

In [ ]:
from huggingface_hub import notebook_login, whoami
notebook_login()          # needs a token with WRITE access — it pushes checkpoints
print("HF user:", whoami()["name"])

In [ ]:
import wandb
wandb.login()             # key from https://wandb.ai/authorize

## 4 — Shared configuration

The holdout is the fiddly part and must match the baselines exactly. The 50 episodes were recorded one
position at a time (`ep 0-4 = P1 … 45-49 = P10`), so lerobot's default "last N episodes" split would hold
out **all of P10** and remove that position from training. Reordering `--dataset.episodes` puts a balanced
holdout in the tail instead: the last episode of P2, P4, P6, P8, P10.

In [ ]:
HF_USER = "HALDijkstraaa"
DATASET = f"{HF_USER}/so101_toolkit_cylinder_20260917_165544"
TRIMMED = f"{DATASET}_trimmed"                  # built in section 7, or pushed from the laptop
PROJECT = "phase06-camera-ablation"             # same project as the baselines, so curves overlay
SEED, STEPS, BATCH = 1000, 60_000, 8

HOLDOUT  = [9, 19, 29, 39, 49]
EPISODES = [e for e in range(50) if e not in HOLDOUT] + HOLDOUT

STATE = "'observation.state': {'type': 'STATE', 'shape': [6]}"
CAM   = lambda c: f"'observation.images.{c}': {{'type': 'VISUAL', 'shape': [3, 480, 640]}}"
FEATS = {"top":   f"{{{STATE}, {CAM('top')}}}",
         "wrist": f"{{{STATE}, {CAM('wrist')}}}",
         "both":  f"{{{STATE}, {CAM('top')}, {CAM('wrist')}}}"}

# Checkpoints go to the Hub as well as to disk, so a Colab disconnect is recoverable (section 9).
def train_cmd(cond, job, dataset=DATASET, steps=STEPS, extra=()):
    return ["lerobot-train",
        f"--dataset.repo_id={dataset}",
        f"--dataset.episodes={EPISODES}",
        "--dataset.eval_split=0.1", "--eval_steps=5000",
        "--policy.type=act", "--policy.device=cuda",
        f"--policy.input_features={FEATS[cond]}",
        "--policy.push_to_hub=true", f"--policy.repo_id={HF_USER}/{job}", "--policy.private=true",
        "--save_checkpoint_to_hub=true",
        f"--batch_size={BATCH}", f"--steps={steps}", "--num_workers=8", f"--seed={SEED}",
        "--save_freq=10000", "--log_freq=200",
        f"--output_dir=/content/outputs/{job}", f"--job_name={job}",
        "--wandb.enable=true", f"--wandb.project={PROJECT}", *extra]

import subprocess, sys, time
def run(cmd):
    t0 = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
    p.wait()
    print(f"\n[exit {p.returncode} after {(time.time() - t0) / 3600:.2f} h]")
    return p.returncode

print("holdout:", HOLDOUT, "| episode-list tail:", EPISODES[-6:])

## 5 — Measure this GPU (≈5 min)

300 steps of the two-camera config, to replace the estimates above with your actual numbers.

In [ ]:
import re, shutil
shutil.rmtree("/content/outputs/bench", ignore_errors=True)
cmd = [c for c in train_cmd("both", "bench", steps=300)
       if not c.startswith(("--policy.push_to_hub", "--policy.repo_id", "--policy.private",
                            "--save_checkpoint_to_hub", "--wandb.enable", "--eval_steps",
                            "--dataset.eval_split"))]
cmd += ["--policy.push_to_hub=false", "--wandb.enable=false",
        "--save_freq=100000", "--eval_steps=0", "--dataset.eval_split=0"]
out = subprocess.run(cmd, capture_output=True, text=True)
text = out.stdout + out.stderr
rates = [float(x) for x in re.findall(r"([\d.]+)step/s", text)]
if rates:
    r2 = sorted(rates)[len(rates) // 2]      # median progress-bar rate, two cameras
    r1 = r2 * 1.95                            # one camera is ~2x (locally 134 vs 72 ms/step)
    print(f"\nmeasured: {r2:.1f} it/s two-camera, ~{r1:.1f} it/s single-camera")
    print(f"  F2 (1 x 60k, two-camera): {STEPS / r2 / 3600:.1f} h")
    print(f"  F3 (top + wrist + both) : {STEPS / r1 / 3600:.1f} + {STEPS / r1 / 3600:.1f} "
          f"+ {STEPS / r2 / 3600:.1f} = {(2 * STEPS / r1 + STEPS / r2) / 3600:.1f} h")
else:
    print("could not parse a rate — output tail follows"); print(text[-3000:])
shutil.rmtree("/content/outputs/bench", ignore_errors=True)

---
# F2 — a learned per-camera identity embedding

In lerobot's ACT every camera goes through **one shared ResNet18**, and `encoder_cam_feat_pos_embed` is a
2D sinusoidal embedding of the feature map's H×W — so `top`'s 300 tokens and `wrist`'s 300 tokens get
**identical positional embeddings**. The model must infer which camera a token came from purely from
appearance, then learn phase-dependent trust, from 45 demonstrations.

The patch adds `nn.Embedding(n_cameras, dim_model)` and adds the right vector to each camera's block.
**Zero-initialised**, so at step 0 the model is exactly upstream ACT — it can only add capacity, never
perturb the starting point. Cost: 2 × 512 = **1024 parameters**.

*(Same code as `scripts/patch_act_camera_embed.py` in the repo. Verified locally: params land at
51,597,190 + 1,024 and a forward pass runs.)*

In [ ]:
import sys
from pathlib import Path
import lerobot.policies.act.modeling_act as m

f = Path(m.__file__); src = f.read_text()

A_OLD = (
    "        if self.config.image_features:\n"
    "            self.encoder_img_feat_input_proj = nn.Conv2d(\n"
    "                backbone_model.fc.in_features, config.dim_model, kernel_size=1\n"
    "            )\n")
A_NEW = A_OLD + (
    "            # F2: learned identity vector per camera, zero-init so step 0 == upstream ACT.\n"
    "            self.camera_id_embed = nn.Embedding(len(self.config.image_features), config.dim_model)\n"
    "            nn.init.zeros_(self.camera_id_embed.weight)\n")

B_OLD = (
    "            for img in batch[OBS_IMAGES]:\n"
    '                cam_features = self.backbone(img)["feature_map"]\n'
    "                cam_pos_embed = self.encoder_cam_feat_pos_embed(cam_features).to(dtype=cam_features.dtype)\n"
    "                cam_features = self.encoder_img_feat_input_proj(cam_features)\n")
B_NEW = (
    "            for cam_idx, img in enumerate(batch[OBS_IMAGES]):\n"
    '                cam_features = self.backbone(img)["feature_map"]\n'
    "                cam_pos_embed = self.encoder_cam_feat_pos_embed(cam_features).to(dtype=cam_features.dtype)\n"
    "                cam_features = self.encoder_img_feat_input_proj(cam_features)\n"
    "                # F2: tag the block with which camera it came from (broadcasts over h, w)\n"
    "                cam_features = cam_features + self.camera_id_embed.weight[cam_idx].view(1, -1, 1, 1)\n")

if "camera_id_embed" in src:
    print("already patched")
else:
    for old, new, where in ((A_OLD, A_NEW, "__init__"), (B_OLD, B_NEW, "forward")):
        assert src.count(old) == 1, f"anchor not found exactly once in {where} - lerobot version mismatch"
        src = src.replace(old, new)
    f.write_text(src)
    print("patched", f)

print("\n!! Restart the runtime now (Runtime > Restart session), then re-run sections 3 and 4 and "
      "continue from the verification cell. Python still holds the unpatched module in memory.")

### Verification — run **after** restarting the runtime and re-running sections 3 and 4

In [ ]:
import torch
from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.policies.act.modeling_act import ACTPolicy
from lerobot.configs.types import FeatureType, PolicyFeature as PF

cfg = ACTConfig(device="cpu")
cfg.input_features = {"observation.state": PF(type=FeatureType.STATE, shape=(6,)),
                      "observation.images.top": PF(type=FeatureType.VISUAL, shape=(3, 480, 640)),
                      "observation.images.wrist": PF(type=FeatureType.VISUAL, shape=(3, 480, 640))}
cfg.output_features = {"action": PF(type=FeatureType.ACTION, shape=(6,))}
p = ACTPolicy(cfg)
n = sum(x.numel() for x in p.parameters())
assert n == 51_597_190 + 1024, f"expected baseline + 1024, got {n}"
assert bool((p.model.camera_id_embed.weight == 0).all()), "embedding must start at zero"
print(f"OK  params {n:,} = 51,597,190 + 1,024   camera_id_embed "
      f"{tuple(p.model.camera_id_embed.weight.shape)} zero-init")

### F2 training run

In [ ]:
run(train_cmd("both", f"act_both_s{SEED}_camemb"))

---
# F3 — retrain on idle-trimmed data

Training episodes were never idle-trimmed: the arm sits still for a **median of 74 frames (2.5 s)** before
it first moves, and 50/50 episodes idle longer than 25 frames. So ACT at the home pose predicts a chunk
that starts with "stay still" in every episode it learned from, and any short action chunk stalls. Only
the full 100-step chunk clears the lead-in — which costs re-observation rate, and with it the sensitivity
of the whole camera ablation.

> **Faster: build the trimmed set on the laptop instead.** The trim is pure CPU — ~40 min on the 24-core
> machine versus 1–1.5 h of a paid GPU session here:
> ```
> python scripts/make_trimmed_dataset.py <src_repo> <dst_repo> ~/trimmed
> ```
> then push `~/trimmed` to the Hub and skip to section 8.

## 7 — Build the trimmed dataset (one-off; skip if it is already on the Hub)

In [ ]:
# Same logic as scripts/make_trimmed_dataset.py. Verified locally on a 3-episode set: parquet rows ==
# decoded video frames for both cameras, and every trimmed episode already shows motion in its first
# 30 frames. Roughly 13 frames/s on a 24-core box, slower here.
import glob, shutil, time, numpy as np, pandas as pd, torch
from pathlib import Path
from lerobot.datasets.lerobot_dataset import LeRobotDataset

THRESH, PAD, ROOT = 2.0, 5, Path("/content/trimmed")
t0 = time.time()
src = LeRobotDataset(DATASET)
print(f"source: {src.num_episodes} episodes, {src.num_frames} frames")
if ROOT.exists(): shutil.rmtree(ROOT)

# States from the parquet — reading them through src[i] would decode both videos for every frame.
raw = pd.concat([pd.read_parquet(f) for f in
                 sorted(glob.glob(str(Path(src.root) / "data/chunk-*/*.parquet")))]).sort_values("index")
states = np.stack(raw["observation.state"].to_numpy())

feats = {k: v for k, v in src.features.items()
         if k not in ("index", "episode_index", "frame_index", "timestamp", "task_index")}
dst = LeRobotDataset.create(TRIMMED, fps=src.fps, features=feats, root=ROOT,
                            robot_type=src.meta.robot_type, use_videos=True)
img_keys = [k for k in feats if k.startswith("observation.images")]

kept = 0
for ep in range(src.num_episodes):
    lo = int(src.meta.episodes["dataset_from_index"][ep])
    hi = int(src.meta.episodes["dataset_to_index"][ep])
    st = states[lo:hi]
    dev0 = np.abs(st - st[0]).max(axis=1)        # departure from the start pose
    dev1 = np.abs(st - st[-1]).max(axis=1)       # departure from the final pose
    a = max(0, (int(np.argmax(dev0 > THRESH)) if (dev0 > THRESH).any() else 0) - PAD)
    back = np.where(dev1 > THRESH)[0]
    b = min(len(st), (int(back[-1]) + 1 if len(back) else len(st)) + PAD)
    for i in range(lo + a, lo + b):              # only kept frames get decoded
        item = src[i]
        frame = {"task": item["task"]}
        for k in feats:
            v = item[k]
            if k in img_keys:                     # CHW float [0,1] -> HWC uint8
                v = (v.permute(1, 2, 0).numpy() * 255).round().clip(0, 255).astype(np.uint8)
            elif isinstance(v, torch.Tensor):
                v = v.numpy()
            frame[k] = v
        dst.add_frame(frame)
    dst.save_episode()
    kept += b - a
    print(f"  ep{ep:02d}: {hi - lo:4d} -> {b - a:4d}  (cut {a} lead-in, {len(st) - b} tail)", flush=True)

print(f"kept {kept}/{src.num_frames} ({100 * kept / src.num_frames:.0f}%) in {(time.time() - t0) / 60:.0f} min")
dst.push_to_hub(private=True)
print("pushed", TRIMMED)

## 8 — F3 training runs

Three runs, sequentially, one per cell so a disconnect only costs the run in flight. **Steps stay at
60k**, not epochs — the trimmed set has ~17% fewer frames, so these see slightly more passes than the
baselines. That is the right call for comparability: same optimisation budget, different data.

In [ ]:
run(train_cmd("top", f"act_top_s{SEED}_trim", dataset=TRIMMED))

In [ ]:
run(train_cmd("wrist", f"act_wrist_s{SEED}_trim", dataset=TRIMMED))

In [ ]:
run(train_cmd("both", f"act_both_s{SEED}_trim", dataset=TRIMMED))

---
## 9 — Resuming after a disconnect

Every run pushes checkpoints to its Hub repo, so nothing is lost. lerobot downloads the highest-numbered
checkpoint into a fresh local dir and continues, keeping the same wandb run:

```python
run(["lerobot-train", f"--config_path={HF_USER}/act_both_s1000_camemb",
     "--resume=true", f"--steps={STEPS}", "--output_dir=/content/outputs/resumed"])
```

Step counter, optimizer state, episode order, holdout and seed all come back from the checkpoint; only
`--steps` is read from the CLI.

## 10 — What to compare

Everything lands in the **`phase06-camera-ablation`** wandb project next to the baselines, so `eval_loss`
overlays directly.

| Compare | Against | Reading |
|---|---|---|
| `act_both_s1000_camemb` | `act_both_s1000` | A lower held-out loss is *encouraging but not the answer* — the baseline's deficit was in reach behavior, which action L1 barely sees. The real test is the robot |
| `act_*_s1000_trim` | `act_*_s1000` | Expect a **higher** held-out loss: the trimmed set has no trivial "stay still" frames left to predict. That is not a regression |

**Both experiments end at the robot, not at the loss curve.** Pull the checkpoints with `hf download`,
then run the §8 retest protocol — interleaved against the relevant baseline in one session, scoring
*reached the cylinder* rather than success, and for F3 with a short `--policy.n_action_steps` (25), which
is the whole point of the trim.